In [1]:
!git clone https://github.com/pyther-hub/ml4sci-faseroh-POC.git

Cloning into 'ml4sci-faseroh-POC'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 138 (delta 85), reused 89 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 162.57 KiB | 6.25 MiB/s, done.
Resolving deltas: 100% (85/85), done.


In [2]:
%cd /kaggle/working/ml4sci-faseroh-POC/

/kaggle/working/ml4sci-faseroh-POC


In [3]:

import json
import sys
from dataclasses import dataclass, fields
from pathlib import Path

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import integrate as sci_integrate
from torch.utils.data import DataLoader
from dataset import FASeROHDataset, collate_fn          # noqa: E402
from dataset_generation import generate_histogram, goodness_of_fit  # noqa: E402
from inference import run_inference                                  # noqa: E402
from metrics import evaluate_predictions, r2_score, _pred_fn_from_tokens  # noqa: E402
from model import FASeROH                               # noqa: E402
from  train import train_one_epoch, evaluate, _print_sample  # noqa: E402

# ── 2. Configuration ──────────────────────────────────────────────────────────

import os

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    DATASET_JSON_PATH: str = "/kaggle/input/datasets/tensorpanda231/faseroh-datasets/dataset_demo_100k.json"
    print("Running in a Kaggle notebook environment.")
else:
    DATASET_JSON_PATH: str = "data/dataset_demo_100k.json"
    print("Not running in a Kaggle notebook environment.")
    
OPTIMISE_FOR_FLOAT: bool = True  # If True, adds MSE mantissa loss with lambda warmup alongside CE


Running in a Kaggle notebook environment.


In [4]:


@dataclass
class FASeROHConfig:
    # Dataset splits
    train_frac: float = 0.9
    val_frac: float = 0.09999
    test_frac: float = 1-train_frac-val_frac

    # Vocabulary
    max_seq_len: int = 128
    pad_token: str = "<pad>"
    sos_token: str = "<sos>"
    eos_token: str = "<eos>"

    # Model architecture
    d_model: int = 256
    n_heads: int = 16
    n_enc_layers: int = 4
    n_dec_layers: int = 6
    n_latent: int = 32
    use_conv: bool = True
    conv_kernel: int = 3
    dropout: float = 0.1

    # Training
    batch_size: int = 256
    lr: float = 3e-4
    n_epochs: int = 100
    evaluate_after: int = 1
    lambda_const: float = 0.1
    lambda_warmup_epochs: int = 5
    grad_clip: float = 1.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint_path: str = "checkpoints/best_model.pt"
    log_every_n_steps: int = 50

    # Inference
    top_k: int = 5
    n_inference_samples: int = 100
    refine_constants: bool = True
    n_refine_candidates: int = 5
    refine_max_iter: int = 100


config = FASeROHConfig()

# ── Interactive configuration ────────────────────────────────────────────────
# Allow user to override any config field before training starts.

_field_names = {f.name for f in fields(FASeROHConfig)}

print("\n" + "=" * 55)
print("  FASEROH POC — Current Configuration")
print("=" * 55)
for k, v in vars(config).items():
    print(f"  {k:30s} = {v}")
print("=" * 55)


# ── 3. Helper functions ───────────────────────────────────────────────────────

def _load_json_records(path: str) -> list[dict]:
    with open(path) as f:
        raw = json.load(f)
    records = []
    for rec in raw:
        tokens = rec.get("encoding", {}).get("tokens", [])
        # Skip records with removed tokens
        if "tan" in tokens or "abs" in tokens:
            continue
        # Migrate exp → e for Euler's constant
        rec["encoding"]["tokens"] = ["e" if t == "exp" else t for t in tokens]
        if "expr_str" in rec:
            rec["expr_str"] = rec["expr_str"].replace("exp", "e")
        hist = rec["histogram"]
        hist["bins"] = np.array(hist["bins"], dtype=int)
        rec["histogram"] = hist
        records.append(rec)
    return records


def _split_records(
    records: list[dict], config: FASeROHConfig,
) -> tuple[list[dict], list[dict], list[dict]]:
    total = len(records)
    n_train = int(config.train_frac * total)
    n_val = int(config.val_frac * total)
    n_test = total - n_train - n_val
    if n_test <= 0:
        raise ValueError(
            f"Dataset has {total} records but train_frac={config.train_frac} "
            f"and val_frac={config.val_frac} leave no records for the test set."
        )
    return (
        records[:n_train],
        records[n_train : n_train + n_val],
        records[n_train + n_val :],
    )


def _build_dataloaders(
    train_recs: list[dict],
    val_recs: list[dict],
    test_recs: list[dict],
    config: FASeROHConfig,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    train_ds = FASeROHDataset(train_recs, config)
    val_ds = FASeROHDataset(val_recs, config)
    test_ds = FASeROHDataset(test_recs, config)

    train_loader = DataLoader(
        train_ds, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_ds, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn,
    )
    return train_loader, val_loader, test_loader


def _numpy_fn_from_expr_str(expr_str: str):
    _safe_ns: dict = {
        "sin": np.sin, "cos": np.cos, "e": np.e,
        "sqrt": np.sqrt, "log": np.log, "abs": np.abs,
        "pi": np.pi, "__builtins__": {},
    }
    def fn(x: np.ndarray) -> np.ndarray:
        ns = dict(_safe_ns)
        ns["x"] = x
        return eval(expr_str, ns)  # noqa: S307
    return fn


# ── 4. Data loading ───────────────────────────────────────────────────────────

print(f"Loading dataset from {DATASET_JSON_PATH!r} ...")
records = _load_json_records(DATASET_JSON_PATH)
train_recs, val_recs, test_recs = _split_records(records, config)
print(
    f"  Total={len(records)}  "
    f"train={len(train_recs)}  val={len(val_recs)}  test={len(test_recs)}"
)

train_loader, val_loader, test_loader = _build_dataloaders(
    train_recs, val_recs, test_recs, config,
)

numpy_fns = [_numpy_fn_from_expr_str(rec["expr_str"]) for rec in test_recs]


# ── 5. Model setup ────────────────────────────────────────────────────────────

model = FASeROH(config).to(config.device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters : {n_params:,}")
print(f"Device           : {config.device}")


# ── 6. Training loop ──────────────────────────────────────────────────────────

optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.n_epochs, eta_min=1e-6,
)
best_val_loss = float("inf")

ckpt_dir = Path(config.checkpoint_path).parent
ckpt_dir.mkdir(parents=True, exist_ok=True)

# History dict to store all metrics per epoch
history = {
    "epoch": [],
    "train_loss": [],
    "train_sent_acc": [],
    "val_loss": [],
    "val_sent_acc": [],
    "val_tok_acc": [],
    "lr": [],
}


  FASEROH POC — Current Configuration
  train_frac                     = 0.9
  val_frac                       = 0.09999
  test_frac                      = 9.999999999982245e-06
  max_seq_len                    = 128
  pad_token                      = <pad>
  sos_token                      = <sos>
  eos_token                      = <eos>
  d_model                        = 256
  n_heads                        = 16
  n_enc_layers                   = 4
  n_dec_layers                   = 6
  n_latent                       = 32
  use_conv                       = True
  conv_kernel                    = 3
  dropout                        = 0.1
  batch_size                     = 256
  lr                             = 0.0003
  n_epochs                       = 100
  evaluate_after                 = 1
  lambda_const                   = 0.1
  lambda_warmup_epochs           = 5
  grad_clip                      = 1.0
  device                         = cuda
  checkpoint_path                = checkpoi

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Model parameters : 10,164,003
Device           : cuda


In [5]:
import time

for epoch in range(config.n_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{config.n_epochs}")
    print(f"{'='*60}")
    start_time = time.time()

    train_loss, train_sent_acc = train_one_epoch(
        model, train_loader, optimizer, config, epoch,
        optimise_for_float=OPTIMISE_FOR_FLOAT,
    )
    val_loss, val_sent_acc, val_tok_acc = evaluate(model, val_loader, config)
    total_time = time.time() - start_time
    print(f"Total time taken for training and validation {total_time/60} min")

    saved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), config.checkpoint_path)
        saved = " [saved]"

    lr_now = scheduler.get_last_lr()[0]
    print(
        f"  train_loss={train_loss:.4f}  train_sent_acc={train_sent_acc:.4f}  "
        f"val_loss={val_loss:.4f}  val_sent_acc={val_sent_acc:.4f}  "
        f"val_tok_acc={val_tok_acc:.4f}  best_val={best_val_loss:.4f}  "
        f"lr={lr_now:.2e}{saved}"
    )
    scheduler.step()

    # Store metrics for this epoch
    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_sent_acc"].append(train_sent_acc)
    history["val_loss"].append(val_loss)
    history["val_sent_acc"].append(val_sent_acc)
    history["val_tok_acc"].append(val_tok_acc)
    history["lr"].append(lr_now)

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")





Epoch 1/100
    step   50/352  loss=0.8036
    step  100/352  loss=0.5773
    step  150/352  loss=0.5148
    step  200/352  loss=0.4653
    step  250/352  loss=0.4689
    step  300/352  loss=0.4376
    step  350/352  loss=0.4383
Total time taken for training and validation 3.168954328695933 min
  train_loss=0.6750  train_sent_acc=0.0475  val_loss=0.4135  val_sent_acc=0.0873  val_tok_acc=0.8299  best_val=0.4135  lr=3.00e-04 [saved]

Epoch 2/100
    step   50/352  loss=0.4214
    step  100/352  loss=0.3987
    step  150/352  loss=0.4064
    step  200/352  loss=0.3844
    step  250/352  loss=0.4196
    step  300/352  loss=0.4005
    step  350/352  loss=0.4023
Total time taken for training and validation 3.131260573863983 min
  train_loss=0.4146  train_sent_acc=0.0831  val_loss=0.3841  val_sent_acc=0.1114  val_tok_acc=0.8378  best_val=0.3841  lr=3.00e-04 [saved]

Epoch 3/100
    step   50/352  loss=0.4040
    step  100/352  loss=0.4134
    step  150/352  loss=0.3855
    step  200/352  los

In [6]:
# ── 9. Training curves ──────────────────────────────────────────────────────

PLOT_PATH = "/kaggle/working/training_curves.png"

epochs = history["epoch"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("FASEROH Training Curves", fontsize=16, fontweight="bold")

# (a) Loss
ax = axes[0, 0]
ax.plot(epochs, history["train_loss"], label="Train Loss", marker="o", markersize=3)
ax.plot(epochs, history["val_loss"], label="Val Loss", marker="s", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Sentence accuracy
ax = axes[0, 1]
ax.plot(epochs, history["train_sent_acc"], label="Train Sent Acc", marker="o", markersize=3)
ax.plot(epochs, history["val_sent_acc"], label="Val Sent Acc", marker="s", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Sentence Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Val token accuracy
ax = axes[1, 0]
ax.plot(epochs, history["val_tok_acc"], label="Val Token Acc", color="tab:green", marker="^", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Validation Token Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)

# (d) Learning rate
ax = axes[1, 1]
ax.plot(epochs, history["lr"], label="Learning Rate", color="tab:red", marker="d", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.set_title("Learning Rate Schedule")
ax.set_yscale("log")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nTraining curves saved to {PLOT_PATH}")



Training curves saved to /kaggle/working/training_curves.png


In [7]:
# ── 7. Final evaluation ───────────────────────────────────────────────────────

print("\n" + "=" * 55)
print("  Final evaluation — best checkpoint")
print("=" * 55)
state = torch.load(config.checkpoint_path, weights_only=True)
model.load_state_dict(state)
final_metrics = evaluate_predictions(model, test_loader, config, numpy_fns, test_recs)
print(f"\nFinal metrics: {final_metrics}")


# ── 8. Custom function inference ──────────────────────────────────────────────
# For each function: normalize to a valid PDF over [0,1] following the same
# pipeline as dataset_generation (positive, integrates to 1), then generate
# a histogram and run inference.

functions = [
    lambda x: x**3 + 2*x*np.sin(np.pi*x),
    lambda x: np.exp(-x) * np.cos(2*np.pi*x),
    lambda x: np.log(x + 1) + x**2*np.sin(x),
    lambda x: np.sqrt(x + 0.1) * np.cos(np.pi*x),
    lambda x: (x**2 + 1) * np.sin(3*x),
]

print("\n" + "=" * 55)
print("  Custom function inference")
print("=" * 55)

_x_grid_r2 = np.linspace(1e-6, 1 - 1e-6, 100)
n_invalid_custom = 0

for i, fn in enumerate(functions):
    print(fn)
    x_grid = np.linspace(1e-6, 1 - 1e-6, 500)
    ys = fn(x_grid)

    # Shift to non-negative (pipeline requires f(x) >= 0 over [0,1])
    min_y = float(ys.min())
    shift = -min_y if min_y < 0 else 0.0
    fn_pos = (lambda f, s: lambda x: f(x) + s)(fn, shift)

    # Normalize to integrate to 1 over [0,1]
    I, _ = sci_integrate.quad(fn_pos, 0.0, 1.0, limit=200, epsabs=1e-7, epsrel=1e-7)
    fn_normed = (lambda f, norm: lambda x: f(x) / norm)(fn_pos, I)

    # Generate histogram using the same pipeline (Algorithm 1)
    histogram = generate_histogram(fn_normed)

    # Build histogram tensor: shape (1, K, 1), normalized as bins / N
    bins = histogram["bins"].astype(float)
    N = float(histogram["N"])
    normed_bins = bins / N if N > 0 else bins
    hist_tensor = torch.tensor(normed_bins, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)

    # Run inference
    result = run_inference(model, hist_tensor, fn_normed, config)
    best = result["best"]

    is_invalid = (
        "<unk>" in best.get("tokens", [])
        or best.get("prefix_error") is not None
        or best.get("eval_error") is not None
        or best["expr_str"] in ("(invalid)", "(no valid)")
        or best.get("y_pred") is None
    )

    print(f"\nFunction {i + 1}:")
    if is_invalid:
        n_invalid_custom += 1
    else:
        # R² score
        y_true_r2 = fn_normed(_x_grid_r2)
        r2 = r2_score(y_true_r2, best["y_pred"])

        # Goodness-of-fit
        gof_val = None
        pred_fn = _pred_fn_from_tokens(best["tokens"], best["mantissas"])
        if pred_fn is not None:
            try:
                gof_result = goodness_of_fit(pred_fn, histogram)
                gof_val = gof_result["X_per_ndf"]
            except Exception:
                pass

        print(f"  predicted  : {best['expr_str']}")
        print(f"  MSE        : {best['mse']:.6f}")
        print(f"  R²         : {r2:.4f}")
        if gof_val is not None:
            print(f"  GoF χ²/ndf : {gof_val:.4f}  (≈1 is good fit)")
        else:
            print(f"  GoF χ²/ndf : N/A")

print(f"\n  Invalid: {n_invalid_custom}/{len(functions)} functions had no valid prediction")





  Final evaluation — best checkpoint

EVALUATION RESULTS
  Active metrics: ['fn_validity', 'gof', 'prefix_validity', 'r2', 'sentence_acc']
  Samples evaluated     : 1
  R² mean               : -63.1164
  R² median             : -63.1164  (valid: 1/1)
  Sentence accuracy     : 0.0000
  Prefix validity acc   : 1.0000
  Function validity acc : 1.0000  (1 valid, 0 invalid out of 1)
  GoF chi2/ndf mean     : 114.4691  (≈1 is good fit)
  GoF chi2/ndf median   : 114.4691  (1/1 valid fns)

Final metrics: {'r2_mean': -63.11635417522339, 'r2_median': -63.11635417522339, 'sentence_acc': 0.0, 'prefix_validity_acc': 1.0, 'fn_validity_acc': 1.0, 'gof_mean': 114.46908643027045, 'gof_median': 114.46908643027045}

  Custom function inference
<function <lambda> at 0x7e3b4e56c040>

Function 1:
  predicted  : (0.125897 + ((1.19551 * x) + ((0.13926 * (x ** 2)) + (0.537119 * (x ** 3)))))
  MSE        : 0.120986
  R²         : 0.6553
  GoF χ²/ndf : 73.2627  (≈1 is good fit)
<function <lambda> at 0x7e3b4e56c